# VPhot Upload Check

Builds a filename → timestamp table from a directory of FITS lights, so a local
capture session can be checked against VPhot's uploaded-images list (the
"Date/Time" column on VPhot's image list page) to spot frames that failed to
upload.

Timestamps come from each FITS header's `DATE-OBS` (exposure start, UTC) plus
half of `EXPTIME` -- VPhot's Date/Time column shows the **mid-exposure** time,
not the raw exposure-start `DATE-OBS`, confirmed against a real session where
every frame's `DATE-OBS + EXPTIME/2` landed on the exact second VPhot showed
for it. A file with no readable date is still listed, with the problem
recorded in `note` -- never silently dropped, since a corrupt/unreadable local
file is itself a plausible reason an upload would fail.

**One-shot-color (OSC) sessions:** VPhot can show two or three rows sharing the
*same* Date/Time (e.g. a B row and a V row) for what is a single physical
exposure -- these are synthetic per-channel measurements VPhot/ASTAP derived
from one Bayer-matrix raw frame, not separate captures. So the number to check
against is VPhot's count of *distinct* timestamps, not its row count -- a
directory with one raw frame per cadence tick is expected to show fewer VPhot
rows than there are frames if each one got split into multiple filters.

In [1]:
from pathlib import Path

import pandas as pd
from astropy.io import fits

## Function library

In [2]:
def read_fits_row(path, date_keys=("DATE-OBS", "DATE-AVG")):
    """One row of header info for a single FITS file. Never raises -- a file
    that can't be opened or has no usable date keyword still gets a row, with
    the problem recorded in 'note' rather than being dropped.

    timestamp is mid-exposure (date keyword + EXPTIME/2), matching what VPhot
    displays -- not the raw exposure-start value the date keyword itself holds.
    """
    row = {"filename": path.name, "timestamp": None, "timestamp_utc": pd.NaT,
           "exptime": None, "filter": None, "object": None, "note": None}
    try:
        with fits.open(path) as hdul:
            hdr = hdul[0].header
    except Exception as e:
        row["note"] = f"could not open FITS: {type(e).__name__}: {e}"
        return row

    row["exptime"] = hdr.get("EXPTIME", hdr.get("EXPOSURE"))
    row["filter"] = hdr.get("FILTER")
    row["object"] = hdr.get("OBJECT")

    date_str, used_key = None, None
    for k in date_keys:
        if k in hdr and hdr[k]:
            date_str, used_key = hdr[k], k
            break
    if date_str is None:
        row["note"] = f"no usable date keyword ({'/'.join(date_keys)})"
        return row

    try:
        ts_start = pd.to_datetime(date_str)
    except Exception as e:
        row["note"] = f"unparseable {used_key}={date_str!r}: {e}"
        return row

    exptime = row["exptime"] or 0.0
    ts = ts_start + pd.to_timedelta(exptime / 2, unit="s")
    row["timestamp_utc"] = ts
    row["timestamp"] = ts.strftime("%Y-%m-%d %H:%M:%S")   # VPhot's Date/Time format
    return row


def build_fits_table(directory, pattern="*.fit*", recursive=False):
    """Filename -> timestamp table for every FITS file directly in `directory`
    (or under it, if recursive=True). Sorted by timestamp (rows with no
    timestamp sort last), 1-indexed to match VPhot's own row numbering.
    'gap_s' is the time since the previous frame, a quick visual check for a
    capture/cadence break next to any VPhot-side gap.
    """
    folder = Path(directory)
    files = sorted(folder.rglob(pattern) if recursive else folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f"no files matching {pattern!r} in {folder}")

    df = pd.DataFrame([read_fits_row(f) for f in files])
    df = df.sort_values("timestamp_utc", na_position="last").reset_index(drop=True)
    df.index = df.index + 1
    df["gap_s"] = df["timestamp_utc"].diff().dt.total_seconds()
    return df[["filename", "timestamp", "gap_s", "exptime", "filter", "object",
               "note", "timestamp_utc"]]

## Build the table

Point `DIRECTORY` at the folder of lights for one session (e.g. the `Light/`
folder VPhot was uploaded from). `pd.set_option` just keeps long filenames from
being truncated in the display below.

In [3]:
DIRECTORY = "/Users/jwatts/Documents/astrophotography/Photometry/0905/cyaqrfinal/Light"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
table = build_fits_table(DIRECTORY)

n_missing = table["timestamp"].isna().sum()
print(f"{len(table)} files, {n_missing} with no usable timestamp")
table.drop(columns="timestamp_utc")

218 files, 0 with no usable timestamp


,filename,timestamp,gap_s,exptime,filter,object,note
1,Light_cyaqrfinal_60.0s_Bin1_20260904-205738_276deg_0001.fit,2026-09-05 06:57:07,NaN,60.0,None,cyaqrfinal,None
2,Light_cyaqrfinal_60.0s_Bin1_20260904-205839_276deg_0002.fit,2026-09-05 06:58:08,60.799681,60.0,None,cyaqrfinal,None
3,Light_cyaqrfinal_60.0s_Bin1_20260904-205940_276deg_0003.fit,2026-09-05 06:59:09,60.850577,60.0,None,cyaqrfinal,None
4,Light_cyaqrfinal_60.0s_Bin1_20260904-210040_276deg_0004.fit,2026-09-05 07:00:10,60.861814,60.0,None,cyaqrfinal,None
5,Light_cyaqrfinal_60.0s_Bin1_20260904-210141_276deg_0005.fit,2026-09-05 07:01:10,60.752288,60.0,None,cyaqrfinal,None
6,Light_cyaqrfinal_60.0s_Bin1_20260904-210242_276deg_0006.fit,2026-09-05 07:02:11,60.843043,60.0,None,cyaqrfinal,None
7,Light_cyaqrfinal_60.0s_Bin1_20260904-210343_276deg_0007.fit,2026-09-05 07:03:12,60.760216,60.0,None,cyaqrfinal,None
8,Light_cyaqrfinal_60.0s_Bin1_20260904-210444_276deg_0008.fit,2026-09-05 07:04:13,60.821872,60.0,None,cyaqrfinal,None
9,Light_cyaqrfinal_60.0s_Bin1_20260904-210545_276deg_0009.fit,2026-09-05 07:05:14,61.056092,60.0,None,cyaqrfinal,None
10,Light_cyaqrfinal_60.0s_Bin1_20260904-210646_276deg_0010.fit,2026-09-05 07:06:15,60.828728,60.0,None,cyaqrfinal,None


## Flag likely trouble spots

Rows worth a second look before cross-checking against VPhot: no timestamp at
all, or a gap far larger than the session's own cadence (a dropped/failed frame
on the *capture* side, which would explain a missing upload rather than being
caused by one).

In [4]:
GAP_FACTOR = 3.0   # flag a gap this many times the median cadence

median_gap = table["gap_s"].median()
flagged = table[table["timestamp"].isna() |
               (table["gap_s"] > GAP_FACTOR * median_gap)]

print(f"median cadence: {median_gap:.1f} s")
if len(flagged):
    print(f"{len(flagged)} row(s) flagged:")
    display(flagged.drop(columns="timestamp_utc"))
else:
    print("nothing flagged")

median cadence: 60.9 s
1 row(s) flagged:


,filename,timestamp,gap_s,exptime,filter,object,note
176,Light_cyaqrfinal_60.0s_Bin1_20260905-001014_96deg_0176.fit,2026-09-05 10:09:43,755.47458,60.0,None,cyaqrfinal,None


## Export (optional)

Save alongside a copy of VPhot's own list (e.g. pasted into a CSV) so the two
can be diffed directly.

In [5]:
OUT_PATH = None  # e.g. "cyaqr_2026-09-05_local_files.csv" -- set to save

if OUT_PATH:
    table.drop(columns="timestamp_utc").to_csv(OUT_PATH, index=False)
    print(f"saved {OUT_PATH}")